# 2-1절 연습 문제 풀이

이 노트북은 2-1절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch02/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

## 연습 2-1

[그림 2-2]의 퍼셉트론에서 가중치가 각각 0.5와 -0.5, 편향이 0.1일 때, 입력 (1, 0)과 (0, 1)에 대한 뉴런의 가중합 z를 계산해 보자.

In [ ]:
# z = w1*x1 + w2*x2 + b
def weighted_sum(x1, x2, w1=0.5, w2=-0.5, b=0.1):
    return w1 * x1 + w2 * x2 + b

for x1, x2 in [(1, 0), (0, 1)]:
    print(f'입력 ({x1}, {x2}) -> z = {weighted_sum(x1, x2):.1f}')

입력 (1, 0)은 z = 0.5×1 + (-0.5)×0 + 0.1 = **0.6**, 입력 (0, 1)은 z = 0.5×0 + (-0.5)×1 + 0.1 = **-0.4**이다. 가중치의 부호가 다르므로 어느 입력이 1인지에 따라 가중합의 부호가 갈린다.

## 연습 2-2

[그림 2-2]의 퍼셉트론에서 시그모이드 활성화 함수를 거친 출력값 y의 값을 반환하는 파이썬 함수를 작성한 후, [연습 문제 2-1]의 조건을 사용해 퍼셉트론의 출력값을 구해 보자.

In [ ]:
import math

def perceptron(x1, x2, w1=0.5, w2=-0.5, b=0.1):
    z = w1 * x1 + w2 * x2 + b
    return 1 / (1 + math.exp(-z))        # 시그모이드 활성화 함수

for x1, x2 in [(1, 0), (0, 1)]:
    y = perceptron(x1, x2)
    print(f'입력 ({x1}, {x2}) -> y = {y:.4f}')

# 파이토치 함수로도 같은 값을 얻는다.
z = torch.tensor([0.6, -0.4])
print(f'torch.sigmoid: {torch.sigmoid(z).tolist()}')

시그모이드는 z가 0보다 크면 0.5보다 큰 값을, 작으면 0.5보다 작은 값을 낸다. z=0.6은 약 0.6457, z=-0.4는 약 0.4013이 되어 각각 1과 0 쪽으로 기운다.

## 연습 2-3

퍼셉트론의 결정 경계는 가중치와 편향에 따라 달라진다. 다음 세 가지 경우에서 결정 경계의 기울기와 x2축 절편을 각각 구하고 결정 경계가 어떻게 달라지는지 설명해 보자.

w1=1, w2=1, b=0

w1=2, w2=1, b=0

w1=1, w2=1, b=1

In [ ]:
# 결정 경계는 z = w1*x1 + w2*x2 + b = 0 인 직선이다.
# x2에 대해 정리하면  x2 = -(w1/w2) * x1 - b/w2
for w1, w2, b in [(1, 1, 0), (2, 1, 0), (1, 1, 1)]:
    slope = -w1 / w2
    intercept = -b / w2
    print(f'w1={w1}, w2={w2}, b={b} -> 기울기 {slope:+.1f}, x2절편 {intercept:+.1f}')

- **w1=1, w2=1, b=0**: 기울기 -1, 절편 0 → 원점을 지나는 직선
- **w1=2, w2=1, b=0**: 기울기 -2, 절편 0 → 원점을 지나되 더 가파른 직선. 가중치의 **비율**이 기울기를 정한다.
- **w1=1, w2=1, b=1**: 기울기 -1, 절편 -1 → 첫 번째와 기울기는 같고 아래로 평행 이동

정리하면 **가중치는 결정 경계의 기울기(방향)를, 편향은 경계의 위치(평행 이동)를** 결정한다.